In [3]:
import pandas as pd
import numpy as np

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(X_train.memory_usage(deep=True).sum() / 1e6, "MB")

407.937479 MB


In [4]:
from optbinning import OptimalBinning

# test satu kolom numerik simpel dulu
x = X_train['AMT_INCOME_TOTAL'].values
y = y_train.values

optb = OptimalBinning(name='AMT_INCOME_TOTAL', dtype='numerical')
optb.fit(x, y)

print("Berhasil!")
print(optb.binning_table.build())

Berhasil!
                           Bin   Count  Count (%)  Non-event  Event  \
0             (-inf, 76477.50)   19093   0.077611      17577   1516   
1         [76477.50, 97854.75)   28939   0.117634      26503   2436   
2        [97854.75, 127530.00)   41600   0.169100      37929   3671   
3       [127530.00, 157671.00)   56994   0.231675      52163   4831   
4       [157671.00, 180045.00)   25912   0.105330      23742   2170   
5       [180045.00, 211716.00)   17732   0.072079      16302   1430   
6       [211716.00, 252787.50)   24460   0.099428      22616   1844   
7       [252787.50, 310950.00)   13638   0.055437      12709    929   
8             [310950.00, inf)   17640   0.071705      16607   1033   
9                      Special       0   0.000000          0      0   
10                     Missing       0   0.000000          0      0   
Totals                          246008   1.000000     226148  19860   

        Event rate       WoE            IV            JS  
0      

In [5]:
from optbinning import OptimalBinning
import pandas as pd

cat_cols = X_train.select_dtypes(include='object').columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
all_features = num_cols + cat_cols

# Dictionary buat nyimpen hasil binning tiap kolom
binning_dict = {}
failed_cols = []

for i, col in enumerate(all_features):
    try:
        dtype = 'categorical' if col in cat_cols else 'numerical'
        optb = OptimalBinning(name=col, dtype=dtype)
        
        x = X_train[col].values
        optb.fit(x, y_train.values)
        
        binning_dict[col] = optb
        
        if (i+1) % 10 == 0:
            print(f"Progress: {i+1}/{len(all_features)}")
            
    except Exception as e:
        print(f"Gagal di kolom {col}: {e}")
        failed_cols.append(col)

print("\nSelesai!")
print(f"Berhasil: {len(binning_dict)}, Gagal: {len(failed_cols)}")

Progress: 10/131
Progress: 20/131
Progress: 30/131
Progress: 40/131
Progress: 50/131
Progress: 60/131
Progress: 70/131
Progress: 80/131
Progress: 90/131
Progress: 100/131
Progress: 110/131
Progress: 120/131
Progress: 130/131

Selesai!
Berhasil: 131, Gagal: 0


In [6]:
def transform_to_woe(df, binning_dict):
    df_woe = pd.DataFrame(index=df.index)
    
    for col, optb in binning_dict.items():
        df_woe[col] = optb.transform(df[col].values, metric='woe')
    
    return df_woe

X_train_woe = transform_to_woe(X_train, binning_dict)
X_test_woe = transform_to_woe(X_test, binning_dict)

print(X_train_woe.shape)
print(X_test_woe.shape)
X_train_woe.head()

C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\4178590569.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_woe[col] = optb.transform(df[col].values, metric='woe')
C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\4178590569.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_woe[col] = optb.transform(df[col].values, metric='woe')
C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\4178590569.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

(246008, 131)
(61503, 131)


C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\4178590569.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_woe[col] = optb.transform(df[col].values, metric='woe')
C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\4178590569.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_woe[col] = optb.transform(df[col].values, metric='woe')
C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\4178590569.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,REGION_POPULATION_RELATIVE,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,...,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,WEEKDAY_APPR_PROCESS_START,ORGANIZATION_TYPE
0,-0.107260,-0.045581,0.041078,0.173267,-0.045439,0.066914,0.299530,-0.051992,0.0,-0.077137,...,-0.054141,-0.031715,-0.01405,0.066678,0.433394,0.074924,0.035525,-0.293566,-0.017581,-0.040140
1,0.053457,-0.045581,0.041078,0.173267,-0.045439,-0.081007,-0.095481,-0.171923,0.0,-0.077137,...,0.112807,0.014267,-0.01405,0.066678,-0.111253,0.074924,0.035525,-0.433439,-0.029953,0.067662
2,0.053457,-0.053162,0.075179,-0.127461,-0.305811,0.139380,-0.077276,-0.051992,0.0,-0.077137,...,-0.054141,0.014267,-0.01405,-0.183202,-0.111253,-0.229161,0.035525,-0.293566,-0.029953,-0.152585
3,0.053457,-0.053162,-0.215214,-0.051143,-0.045439,-0.081007,-0.185116,-0.051992,0.0,-0.077137,...,-0.054141,-0.031715,-0.01405,0.066678,-0.111253,-0.229161,0.035525,-0.293566,-0.029953,-0.152585
4,0.053457,-0.039964,0.041078,-0.127461,-0.045439,-0.081007,-0.223162,-0.051992,0.0,-0.077137,...,-0.054141,0.014267,-0.01405,0.066678,-0.111253,-0.229161,0.035525,0.368355,0.045957,-0.415373


In [7]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(binning_dict, '../models/binning_dict.pkl')

X_train_woe.to_csv('../data/processed/X_train_woe.csv', index=False)
X_test_woe.to_csv('../data/processed/X_test_woe.csv', index=False)

print("Saved!")

Saved!


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_woe, y_train)

# Prediksi probabilitas default
train_pred = log_reg.predict_proba(X_train_woe)[:, 1]
test_pred = log_reg.predict_proba(X_test_woe)[:, 1]

train_auc = roc_auc_score(y_train, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

Train AUC: 0.7695
Test AUC: 0.7687


In [9]:
from scipy.stats import ks_2samp

def calculate_ks(y_true, y_pred):
    df_ks = pd.DataFrame({'target': y_true, 'pred': y_pred})
    good = df_ks[df_ks['target'] == 0]['pred']
    bad = df_ks[df_ks['target'] == 1]['pred']
    ks_stat, _ = ks_2samp(good, bad)
    return ks_stat

train_ks = calculate_ks(y_train, train_pred)
test_ks = calculate_ks(y_test, test_pred)

print(f"Train KS: {train_ks:.4f}")
print(f"Test KS: {test_ks:.4f}")

Train KS: 0.4044
Test KS: 0.4022


In [10]:
coef_df = pd.DataFrame({
    'feature': X_train_woe.columns,
    'coefficient': log_reg.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.head(20))

                            feature  coefficient
58             ANNUITY_INCOME_RATIO    -1.403821
105  POS_MEAN_CNT_INSTALMENT_FUTURE    -1.142650
86            PREV_MEAN_AMT_ANNUITY    -0.990512
121                    FLAG_OWN_CAR    -0.941048
100        INST_MEAN_AMT_INSTALMENT     0.831832
104                       POS_COUNT    -0.800713
120                     CODE_GENDER    -0.796363
112            CC_MEAN_CREDIT_LIMIT     0.776995
84              PREV_COUNT_APPROVED     0.744909
1                  AMT_INCOME_TOTAL     0.744473
90            PREV_MEAN_CNT_PAYMENT    -0.721813
73          BUREAU_TOTAL_CREDIT_SUM    -0.718769
24                     EXT_SOURCE_2    -0.713364
93                       INST_COUNT    -0.692470
68              BUREAU_COUNT_ACTIVE    -0.661200
54        AMT_REQ_CREDIT_BUREAU_QRT    -0.644345
87             PREV_MEAN_AMT_CREDIT     0.632107
25                     EXT_SOURCE_3    -0.611829
114              CC_MAX_UTILIZATION    -0.564154
130               OR

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# L1 regularization (Lasso) - otomatis menekan fitur yang redundant/tidak stabil
log_reg_l1 = LogisticRegression(
    penalty='l1',
    solver='liblinear',  # solver yang support L1
    C=0.1,  # semakin kecil C, semakin kuat regularisasi (semakin banyak fitur ditekan)
    random_state=42,
    max_iter=1000
)
log_reg_l1.fit(X_train_woe, y_train)

train_pred_l1 = log_reg_l1.predict_proba(X_train_woe)[:, 1]
test_pred_l1 = log_reg_l1.predict_proba(X_test_woe)[:, 1]

train_auc_l1 = roc_auc_score(y_train, train_pred_l1)
test_auc_l1 = roc_auc_score(y_test, test_pred_l1)

train_ks_l1 = calculate_ks(y_train, train_pred_l1)
test_ks_l1 = calculate_ks(y_test, test_pred_l1)

print(f"Train AUC: {train_auc_l1:.4f} | Test AUC: {test_auc_l1:.4f}")
print(f"Train KS: {train_ks_l1:.4f} | Test KS: {test_ks_l1:.4f}")

# Cek berapa banyak fitur yang koefisiennya jadi nol (ditekan habis oleh L1)
coef_l1 = log_reg_l1.coef_[0]
print(f"\nFitur dengan koefisien = 0: {(coef_l1 == 0).sum()} dari {len(coef_l1)}")

Train AUC: 0.7692 | Test AUC: 0.7687
Train KS: 0.4044 | Test KS: 0.4034

Fitur dengan koefisien = 0: 62 dari 131


In [12]:
coef_df_l1 = pd.DataFrame({
    'feature': X_train_woe.columns,
    'coefficient': coef_l1
})
coef_df_l1 = coef_df_l1[coef_df_l1['coefficient'] != 0].sort_values('coefficient', key=abs, ascending=False)

print(coef_df_l1.head(25))

                            feature  coefficient
58             ANNUITY_INCOME_RATIO    -1.210383
100        INST_MEAN_AMT_INSTALMENT     1.055448
105  POS_MEAN_CNT_INSTALMENT_FUTURE    -1.032450
84              PREV_COUNT_APPROVED     0.960254
86            PREV_MEAN_AMT_ANNUITY    -0.944404
104                       POS_COUNT    -0.782116
121                    FLAG_OWN_CAR    -0.767230
120                     CODE_GENDER    -0.749639
90            PREV_MEAN_CNT_PAYMENT    -0.723825
73          BUREAU_TOTAL_CREDIT_SUM    -0.722530
24                     EXT_SOURCE_2    -0.712513
93                       INST_COUNT    -0.649665
68              BUREAU_COUNT_ACTIVE    -0.637149
25                     EXT_SOURCE_3    -0.618701
112            CC_MEAN_CREDIT_LIMIT     0.544799
1                  AMT_INCOME_TOTAL     0.521374
130               ORGANIZATION_TYPE    -0.520471
54        AMT_REQ_CREDIT_BUREAU_QRT    -0.496570
81         BUREAU_DEBT_CREDIT_RATIO    -0.489231
114              CC_

In [13]:
# Fitur dengan tanda counterintuitive (positif = harusnya berisiko, tapi logikanya harusnya aman)
sign_problem_features = [
    'INST_MEAN_AMT_INSTALMENT', 'PREV_COUNT_APPROVED', 
    'CC_MEAN_CREDIT_LIMIT', 'AMT_INCOME_TOTAL',
    'PREV_MEAN_AMT_CREDIT'  # kalau masih muncul di run berikutnya
]

X_train_woe_v2 = X_train_woe.drop(columns=sign_problem_features)
X_test_woe_v2 = X_test_woe.drop(columns=sign_problem_features)

log_reg_v2 = LogisticRegression(
    penalty='l1', solver='liblinear', C=0.1, random_state=42, max_iter=1000
)
log_reg_v2.fit(X_train_woe_v2, y_train)

train_pred_v2 = log_reg_v2.predict_proba(X_train_woe_v2)[:, 1]
test_pred_v2 = log_reg_v2.predict_proba(X_test_woe_v2)[:, 1]

print(f"Train AUC: {roc_auc_score(y_train, train_pred_v2):.4f} | Test AUC: {roc_auc_score(y_test, test_pred_v2):.4f}")
print(f"Train KS: {calculate_ks(y_train, train_pred_v2):.4f} | Test KS: {calculate_ks(y_test, test_pred_v2):.4f}")

coef_v2 = log_reg_v2.coef_[0]
coef_df_v2 = pd.DataFrame({
    'feature': X_train_woe_v2.columns,
    'coefficient': coef_v2
})
coef_df_v2 = coef_df_v2[coef_df_v2['coefficient'] != 0].sort_values('coefficient', key=abs, ascending=False)
print(f"\nFitur aktif: {(coef_v2 != 0).sum()}")
print(coef_df_v2.head(30))

Train AUC: 0.7680 | Test AUC: 0.7677
Train KS: 0.4022 | Test KS: 0.4000

Fitur aktif: 68
                            feature  coefficient
57             ANNUITY_INCOME_RATIO    -1.013184
84            PREV_MEAN_AMT_ANNUITY    -0.918110
101  POS_MEAN_CNT_INSTALMENT_FUTURE    -0.896089
115                     CODE_GENDER    -0.780168
87            PREV_MEAN_CNT_PAYMENT    -0.776025
100                       POS_COUNT    -0.763066
116                    FLAG_OWN_CAR    -0.752889
23                     EXT_SOURCE_2    -0.710435
72          BUREAU_TOTAL_CREDIT_SUM    -0.650942
67              BUREAU_COUNT_ACTIVE    -0.647400
24                     EXT_SOURCE_3    -0.623703
125               ORGANIZATION_TYPE    -0.531948
82          PREV_COUNT_APPLICATIONS     0.527024
53        AMT_REQ_CREDIT_BUREAU_QRT    -0.512263
80         BUREAU_DEBT_CREDIT_RATIO    -0.489940
97                  INST_RATIO_LATE    -0.455981
22                     EXT_SOURCE_1    -0.445822
110            CC_MEAN_DRAWIN

In [14]:
final_drop = ['PREV_COUNT_APPLICATIONS', 'INST_SUM_FLAG_SHORTFALL', 'FLAG_EMP_PHONE']

X_train_woe_final = X_train_woe_v2.drop(columns=final_drop)
X_test_woe_final = X_test_woe_v2.drop(columns=final_drop)

log_reg_final = LogisticRegression(
    penalty='l1', solver='liblinear', C=0.1, random_state=42, max_iter=1000
)
log_reg_final.fit(X_train_woe_final, y_train)

train_pred_final = log_reg_final.predict_proba(X_train_woe_final)[:, 1]
test_pred_final = log_reg_final.predict_proba(X_test_woe_final)[:, 1]

print(f"Train AUC: {roc_auc_score(y_train, train_pred_final):.4f} | Test AUC: {roc_auc_score(y_test, test_pred_final):.4f}")
print(f"Train KS: {calculate_ks(y_train, train_pred_final):.4f} | Test KS: {calculate_ks(y_test, test_pred_final):.4f}")

coef_final = log_reg_final.coef_[0]
coef_df_final = pd.DataFrame({
    'feature': X_train_woe_final.columns,
    'coefficient': coef_final
})
coef_df_final = coef_df_final[coef_df_final['coefficient'] != 0].sort_values('coefficient', key=abs, ascending=False)
print(f"\nFitur aktif: {(coef_final != 0).sum()}")

# Cek berapa yang masih bertanda positif (counterintuitive)
positive_coef = coef_df_final[coef_df_final['coefficient'] > 0]
print(f"Fitur masih bertanda positif: {len(positive_coef)}")
print(positive_coef)

Train AUC: 0.7674 | Test AUC: 0.7671
Train KS: 0.4026 | Test KS: 0.4008

Fitur aktif: 63
Fitur masih bertanda positif: 5
                        feature  coefficient
32              FLAG_DOCUMENT_6     0.395980
58                    AGE_YEARS     0.094450
19       REG_CITY_NOT_WORK_CITY     0.050209
3    REGION_POPULATION_RELATIVE     0.012930
103                    CC_COUNT     0.002268


In [15]:
X_train_woe_v3 = X_train_woe_final.drop(columns=['FLAG_DOCUMENT_6'])
X_test_woe_v3 = X_test_woe_final.drop(columns=['FLAG_DOCUMENT_6'])

log_reg_v3 = LogisticRegression(
    penalty='l1', solver='liblinear', C=0.1, random_state=42, max_iter=1000
)
log_reg_v3.fit(X_train_woe_v3, y_train)

train_pred_v3 = log_reg_v3.predict_proba(X_train_woe_v3)[:, 1]
test_pred_v3 = log_reg_v3.predict_proba(X_test_woe_v3)[:, 1]

print(f"Train AUC: {roc_auc_score(y_train, train_pred_v3):.4f} | Test AUC: {roc_auc_score(y_test, test_pred_v3):.4f}")
print(f"Train KS: {calculate_ks(y_train, train_pred_v3):.4f} | Test KS: {calculate_ks(y_test, test_pred_v3):.4f}")

Train AUC: 0.7673 | Test AUC: 0.7672
Train KS: 0.4023 | Test KS: 0.4000


In [16]:
import joblib

joblib.dump(log_reg_v3, '../models/logistic_regression_final.pkl')

# simpan juga list fitur final yang dipakai, penting buat konsistensi pas prediksi data baru nanti
final_features = X_train_woe_v3.columns.tolist()
joblib.dump(final_features, '../models/model_a_features.pkl')

print("Model A saved!")

Model A saved!


In [17]:
import numpy as np

# Parameter scorecard standar industri
base_score = 600
base_odds = 1  # odds 1:1 di base_score
pdo = 20  # points to double the odds

factor = pdo / np.log(2)
offset = base_score - factor * np.log(base_odds)

print(f"Factor: {factor:.4f}")
print(f"Offset: {offset:.4f}")

def calculate_score(log_odds):
    return offset + factor * (-log_odds)  # negatif karena log_odds tinggi = risiko tinggi = skor rendah

# Hitung skor untuk train & test
train_log_odds = log_reg_v3.decision_function(X_train_woe_v3)
test_log_odds = log_reg_v3.decision_function(X_test_woe_v3)

train_scores = calculate_score(train_log_odds)
test_scores = calculate_score(test_log_odds)

print(f"\nTrain score range: {train_scores.min():.0f} - {train_scores.max():.0f}")
print(f"Test score range: {test_scores.min():.0f} - {test_scores.max():.0f}")
print(f"Train score mean: {train_scores.mean():.0f}")

Factor: 28.8539
Offset: 600.0000

Train score range: 560 - 788
Test score range: 571 - 789
Train score mean: 682


In [22]:
def build_scorecard_table(binning_dict, log_reg_model, feature_names, factor, offset, n_features):
    scorecard_rows = []
    skipped = []
    
    for feature, coef in zip(feature_names, log_reg_model.coef_[0]):
        if coef == 0:
            continue
        
        optb = binning_dict[feature]
        table = optb.binning_table.build()
        
        for idx, row in table.iterrows():
            try:
                bin_label = row['Bin']
                # Kalau bin_label bukan string tunggal (misal array/list), skip
                if not isinstance(bin_label, str):
                    continue
                if bin_label in ['Special', 'Missing', 'Totals']:
                    continue
                
                woe = float(row['WoE'])
                points = -(coef * woe * factor) + (offset / n_features)
                
                scorecard_rows.append({
                    'Feature': feature,
                    'Bin': bin_label,
                    'WoE': woe,
                    'Coefficient': coef,
                    'Points': round(points, 1)
                })
            except Exception as e:
                skipped.append((feature, str(e)))
                continue
    
    return pd.DataFrame(scorecard_rows), skipped

n_active_features = (log_reg_v3.coef_[0] != 0).sum()

scorecard_table, skipped_rows = build_scorecard_table(
    binning_dict, log_reg_v3, X_train_woe_v3.columns, factor, offset, n_active_features
)

print(scorecard_table.shape)
print(f"Skipped: {len(skipped_rows)}")
if skipped_rows:
    print(skipped_rows[:10])  # tampilkan 10 contoh error pertama biar kita tau polanya

scorecard_table.sort_values(['Feature', 'Points'], ascending=[True, False]).head(30)

(345, 5)
Skipped: 61
[('CNT_CHILDREN', "could not convert string to float: ''"), ('AMT_CREDIT', "could not convert string to float: ''"), ('AMT_ANNUITY', "could not convert string to float: ''"), ('REGION_POPULATION_RELATIVE', "could not convert string to float: ''"), ('DAYS_REGISTRATION', "could not convert string to float: ''"), ('DAYS_ID_PUBLISH', "could not convert string to float: ''"), ('OWN_CAR_AGE', "could not convert string to float: ''"), ('FLAG_WORK_PHONE', "could not convert string to float: ''"), ('FLAG_PHONE', "could not convert string to float: ''"), ('REGION_RATING_CLIENT_W_CITY', "could not convert string to float: ''")]


,Feature,Bin,WoE,Coefficient,Points
145,AGE_YEARS,"(-inf, 25.76)",-0.446316,0.118942,11.4
146,AGE_YEARS,"[25.76, 28.27)",-0.377462,0.118942,11.1
147,AGE_YEARS,"[28.27, 32.00)",-0.338118,0.118942,11.0
148,AGE_YEARS,"[32.00, 34.86)",-0.235756,0.118942,10.6
149,AGE_YEARS,"[34.86, 38.12)",-0.155493,0.118942,10.4
150,AGE_YEARS,"[38.12, 42.03)",-0.023006,0.118942,9.9
151,AGE_YEARS,"[42.03, 44.83)",0.038379,0.118942,9.7
152,AGE_YEARS,"[44.83, 47.55)",0.082491,0.118942,9.6
153,AGE_YEARS,"[47.55, 50.20)",0.137161,0.118942,9.4
154,AGE_YEARS,"[50.20, 53.43)",0.166426,0.118942,9.3


In [23]:
scorecard_display = scorecard_table.sort_values(['Feature', 'Points'], ascending=[True, False])
print(scorecard_display.head(30))

# Cek juga range poin per fitur - biar tau fitur mana yang paling "berat" pengaruhnya ke skor
feature_range = scorecard_table.groupby('Feature')['Points'].agg(['min', 'max'])
feature_range['range'] = feature_range['max'] - feature_range['min']
feature_range = feature_range.sort_values('range', ascending=False)
print("\n--- Top 15 fitur dengan range poin terbesar (paling berpengaruh) ---")
print(feature_range.head(15))

         Feature                       Bin       WoE  Coefficient  Points
145    AGE_YEARS             (-inf, 25.76) -0.446316     0.118942    11.4
146    AGE_YEARS            [25.76, 28.27) -0.377462     0.118942    11.1
147    AGE_YEARS            [28.27, 32.00) -0.338118     0.118942    11.0
148    AGE_YEARS            [32.00, 34.86) -0.235756     0.118942    10.6
149    AGE_YEARS            [34.86, 38.12) -0.155493     0.118942    10.4
150    AGE_YEARS            [38.12, 42.03) -0.023006     0.118942     9.9
151    AGE_YEARS            [42.03, 44.83)  0.038379     0.118942     9.7
152    AGE_YEARS            [44.83, 47.55)  0.082491     0.118942     9.6
153    AGE_YEARS            [47.55, 50.20)  0.137161     0.118942     9.4
154    AGE_YEARS            [50.20, 53.43)  0.166426     0.118942     9.3
155    AGE_YEARS            [53.43, 56.51)  0.338830     0.118942     8.7
156    AGE_YEARS            [56.51, 63.51)  0.437875     0.118942     8.3
157    AGE_YEARS              [63.51, 

In [24]:
scorecard_table.to_csv('../data/processed/scorecard_table.csv', index=False)
print("Scorecard table saved!")

Scorecard table saved!


In [25]:
# Bikin bucket skor, cek default rate aktualnya per bucket
df_calib = pd.DataFrame({'score': test_scores, 'actual_target': y_test.values})
df_calib['score_bucket'] = pd.qcut(df_calib['score'], q=10, duplicates='drop')

calib_table = df_calib.groupby('score_bucket').agg(
    count=('actual_target', 'count'),
    actual_default_rate=('actual_target', 'mean'),
    mean_score=('score', 'mean')
).reset_index()

print(calib_table)

         score_bucket  count  actual_default_rate  mean_score
0  (571.326, 643.585]   6151             0.279142  629.101534
1  (643.585, 657.203]   6150             0.151382  650.836402
2   (657.203, 667.22]   6150             0.100650  662.415205
3   (667.22, 675.624]   6150             0.083740  671.513812
4   (675.624, 683.47]   6151             0.062591  679.604167
5   (683.47, 691.077]   6150             0.042764  687.309977
6  (691.077, 699.129]   6150             0.029919  695.044266
7  (699.129, 708.077]   6150             0.024715  703.411017
8  (708.077, 720.343]   6150             0.020325  713.777655
9  (720.343, 789.039]   6151             0.012031  732.720185


C:\Users\kn409\AppData\Local\Temp\ipykernel_10672\297545770.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib_table = df_calib.groupby('score_bucket').agg(


In [26]:
cutoffs = np.percentile(test_scores, [10, 20, 30, 40, 50, 60, 70, 80, 90])

cutoff_analysis = []
for cutoff in cutoffs:
    approved = test_scores >= cutoff
    approval_rate = approved.mean()
    default_rate_approved = y_test.values[approved].mean() if approved.sum() > 0 else 0
    
    cutoff_analysis.append({
        'cutoff_score': round(cutoff, 0),
        'approval_rate': round(approval_rate, 3),
        'default_rate_in_approved': round(default_rate_approved, 4)
    })

cutoff_df = pd.DataFrame(cutoff_analysis)
print(cutoff_df)

   cutoff_score  approval_rate  default_rate_in_approved
0         644.0            0.9                    0.0587
1         657.0            0.8                    0.0471
2         667.0            0.7                    0.0394
3         676.0            0.6                    0.0321
4         683.0            0.5                    0.0259
5         691.0            0.4                    0.0217
6         699.0            0.3                    0.0190
7         708.0            0.2                    0.0162
8         720.0            0.1                    0.0120


In [27]:
calib_table.to_csv('../data/processed/model_a_calibration.csv', index=False)
cutoff_df.to_csv('../data/processed/model_a_cutoff_analysis.csv', index=False)

print("Model A fully documented and saved!")

Model A fully documented and saved!


In [28]:
X_train_woe_v3.to_csv('../data/processed/X_train_woe_final.csv', index=False)
X_test_woe_v3.to_csv('../data/processed/X_test_woe_final.csv', index=False)
print("WOE final data saved!")

WOE final data saved!
